# Train the Puffer Dino environment on Google Colab

This notebook trains the current Dino environment directly in Colab. It deliberately does **not** use Docker or PufferTank. The last cell copies the newest checkpoint to `MyDrive/Dino-weights`, so it survives when Colab deletes the temporary runtime.

Before running: choose **Runtime → Change runtime type → T4 GPU** (or another GPU).

In [ ]:
# Confirm that Colab attached a GPU and show its compute capability.
!nvidia-smi --query-gpu=name,compute_cap,memory.total --format=csv,noheader
!nvcc --version

In [ ]:
# Authorize access to your Google Drive. The checkpoint will be saved here.
from google.colab import drive
from pathlib import Path

drive.mount('/content/drive')
weights_dir = Path('/content/drive/MyDrive/Dino-weights')
weights_dir.mkdir(parents=True, exist_ok=True)
print(f'Checkpoints will be saved to: {weights_dir}')

In [ ]:
# These are the two repositories used for this run.
PUFFERLIB_REPO = 'https://github.com/PufferAI/PufferLib.git'
PUFFERLIB_BRANCH = '4.0'
ENV_REPO = 'https://github.com/hectoragvz/puffer-envs.git'

In [ ]:
# Install the compiler/runtime dependencies required by PufferLib's native CUDA backend.
!apt-get -qq update && apt-get -qq install -y clang ccache libomp5 libomp-dev
!python -m pip install -q --upgrade pip
!git clone --branch {PUFFERLIB_BRANCH} --depth 1 {PUFFERLIB_REPO} /content/PufferLib
%cd /content/PufferLib
!python -m pip install -q -e . nvidia-cudnn-cu12 nvidia-nccl-cu12
!python -c "import torch; print('PyTorch CUDA available:', torch.cuda.is_available()); print('GPU:', torch.cuda.get_device_name(0))"

In [ ]:
# Copy the Dino implementation and its training configuration into PufferLib.
!git clone --depth 1 {ENV_REPO} /content/puffer-envs
!mkdir -p ocean/dino
!cp /content/puffer-envs/envs/dino/dino.h ocean/dino/dino.h
!cp /content/puffer-envs/envs/dino/dino.c ocean/dino/dino.c
!cp /content/puffer-envs/envs/dino/binding.c ocean/dino/binding.c
!cp /content/puffer-envs/envs/dino/dino.ini config/dino.ini
!grep -n -C 2 'JUMP_PENALTY' ocean/dino/dino.h

In [ ]:
# Convert the GPU's compute capability (for example 7.5) into NVCC's architecture name (sm_75).
import subprocess

capability = subprocess.check_output(
    ['nvidia-smi', '--query-gpu=compute_cap', '--format=csv,noheader'], text=True
).splitlines()[0].strip()
nvcc_arch = 'sm_' + capability.replace('.', '')
print(f'Building for compute capability {capability}: {nvcc_arch}')

In [ ]:
# Build the CUDA training extension. Explicit architecture avoids NVCC's -arch=native detection.
!NVCC_ARCH={nvcc_arch} bash build.sh dino

In [ ]:
# Train for the 10,000,000 timesteps specified in config/dino.ini.
!puffer train dino

In [ ]:
# Copy the newest native checkpoint to Drive with a stable filename.
import shutil

checkpoints = list(Path('checkpoints/dino').rglob('*.bin'))
if not checkpoints:
    raise FileNotFoundError('Training produced no .bin checkpoint under checkpoints/dino.')

newest = max(checkpoints, key=lambda path: path.stat().st_mtime)
destination = weights_dir / 'dino_weights.bin'
shutil.copy2(newest, destination)
print(f'Saved checkpoint: {destination}')
print(f'Source checkpoint: {newest}')

In [ ]:
# Optional: download the same checkpoint for resources/dino/dino_weights.bin.
from google.colab import files
files.download(str(destination))